In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import math
import matplotlib.dates as mdates
import matplotlib.ticker as mtick

In [ ]:
df_team_match = pd.read_csv("../OUTPUTS/df_team_match.csv")
df_player_latest_imputed = pd.read_csv("../OUTPUTS/df_player_latest_imputed.csv")
df_player_latest = pd.read_csv("../OUTPUTS/df_player_latest.csv")

df_team_match["DATE"] = pd.to_datetime(df_team_match["DATE"], errors="coerce").dt.floor("s")

df_player_latest["DATE"] = pd.to_datetime(df_player_latest["DATE"], errors="coerce").dt.floor("s")
df_player_latest["BIRTHDAY"] = pd.to_datetime(df_player_latest["BIRTHDAY"], errors="coerce").dt.floor("s")

df_player_latest_imputed["DATE"] = pd.to_datetime(df_player_latest_imputed["DATE"], errors="coerce").dt.floor("s")
df_player_latest_imputed["BIRTHDAY"] = pd.to_datetime(df_player_latest_imputed["BIRTHDAY"], errors="coerce").dt.floor("s")

In [ ]:
COLOR_FONDO = "#F7F7F2"
COLOR_TEXTO = "#1F2937"
COLOR_GRID = "#D0D7DE"

COLOR_PRINCIPAL = "#1D4E89"
COLOR_SECUNDARIO = "#2A9D8F"
COLOR_RESALTE = "#E9C46A"
COLOR_ALERTA = "#D95D39"
COLOR_NEUTRO = "#8D99AE"

PALETA = [
    COLOR_PRINCIPAL,
    COLOR_SECUNDARIO,
    COLOR_RESALTE,
    COLOR_ALERTA,
    COLOR_NEUTRO,
    "#84A59D"
]

CMAP_CORR = "RdBu_r"

plt.rcParams.update({
    "figure.facecolor": COLOR_FONDO,
    "axes.facecolor": COLOR_FONDO,
    "savefig.facecolor": COLOR_FONDO,
    "axes.edgecolor": "#AAB4BE",
    "axes.labelcolor": COLOR_TEXTO,
    "axes.titlecolor": COLOR_TEXTO,
    "xtick.color": COLOR_TEXTO,
    "ytick.color": COLOR_TEXTO,
    "text.color": COLOR_TEXTO,
    "grid.color": COLOR_GRID,
    "grid.alpha": 0.35,
    "font.family": "sans-serif",
    "font.sans-serif": ["Aptos", "Segoe UI", "Calibri", "DejaVu Sans"],
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": 120
})

In [ ]:
# Nos quedamos solo con jugadores de campo
df_outfield = df_player_latest_imputed.copy()
df_outfield = df_outfield[df_outfield["PLAYER_TYPE"] == "OUTFIELD"]

# Seleccionamos atributos relevantes para jugadores de campo
cols = [
    "OVERALL_RATING",
    "POTENTIAL",
    "CROSSING",
    "FINISHING",
    "SHORT_PASSING",
    "DRIBBLING",
    "LONG_PASSING",
    "BALL_CONTROL",
    "ACCELERATION",
    "SPRINT_SPEED",
    "REACTIONS",
    "SHOT_POWER",
    "STAMINA",
    "INTERCEPTIONS",
    "POSITIONING",
    "VISION",
    "STANDING_TACKLE"
]

# Calculamos la matriz de correlacion
df_corr = df_outfield[cols].copy()
corr = df_corr.corr(method = "spearman")

# Preparamos labels mas limpios para el grafico
labels = [col.replace("_", " ") for col in corr.columns]

# Graficamos el heatmap
fig, ax = plt.subplots(figsize = (12, 9))
im = ax.imshow(corr.values, cmap = CMAP_CORR, vmin = -1, vmax = 1)

ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels, rotation = 90, ha = "right")
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        value = corr.values[i, j]
        color = "white" if abs(value) >= 0.60 else "black"
        text = ax.text(j, i, f"{value:.2f}", ha = "center", va = "center", color = color, fontsize = 8)

ax.set_title("Correlacion de atributos de jugadores de campo", loc = "left", pad = 15)
ax.tick_params(length = 0)
cbar = plt.colorbar(im, ax = ax, shrink=0.85)
cbar.set_label("Correlacion de Spearman")

fig.tight_layout()
plt.show()